# Sandile's SectorWatch — Data Science Project
## Energy & Defense Equity Analytics | Iran War 2026 → Present

**Author:** Sandile Kumalo  
**Period:** February 28, 2026 → *today* (always current)  
**Sectors:** Defense (11 stocks) · Energy (9 stocks)  
**Portfolio:** LMT · RTX · NOC · XOM · CVX · COP · SLB *(7 holdings)*

### How the data works
| Source | Coverage | Key |
|--------|----------|-----|
| Yahoo Finance (primary) | Live, always current | None needed |
| Finnhub | Live, always current | Free at finnhub.io |
| Alpha Vantage | Live, always current | Free at alphavantage.co |
| Twelve Data | Live, always current | Free at twelvedata.com |
| Smart fallback | Extends itself to today automatically | Built-in |

> The notebook is **date-aware** — it always fetches from Feb 28, 2026 to **today**, so it never goes stale.

> ⚠️ For educational purposes only. Not investment advice.


## 0 · Setup

In [ ]:
import subprocess, sys
for pkg in ['yfinance','pandas','numpy','scikit-learn','matplotlib','seaborn','statsmodels']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print("✓ All packages ready")


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime, timedelta

from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit
from sklearn.metrics import (classification_report, confusion_matrix,
                              mean_absolute_error, mean_squared_error, r2_score)
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

plt.rcParams.update({
    'figure.facecolor': '#F8FAFC', 'axes.facecolor': '#FFFFFF',
    'axes.edgecolor': '#E2E6EA', 'axes.labelcolor': '#1A2E4A',
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlecolor': '#1A2E4A',
    'xtick.color': '#6B7A8D', 'ytick.color': '#6B7A8D',
    'grid.color': '#E2E6EA', 'grid.alpha': 0.6, 'font.family': 'sans-serif',
    'legend.framealpha': 0.92, 'legend.edgecolor': '#E2E6EA',
    'legend.fontsize': 8, 'figure.dpi': 120,
})

NAVY='#1A2E4A'; BLUE='#2E6DA4'; BLUE2='#378ADD'
AMBER='#BA7517'; RED='#A32D2D'; GREEN='#3B6D11'
PORTFOLIO = ['LMT','RTX','NOC','XOM','CVX','COP','SLB']

# Always anchored to war start; end date = TODAY automatically
WAR_START = pd.Timestamp('2026-02-28')
TODAY     = pd.Timestamp(datetime.today().date())

print(f"✓ Libraries loaded")
print(f"  War start : {WAR_START.date()}")
print(f"  Today     : {TODAY.date()}")
print(f"  Coverage  : {(TODAY - WAR_START).days} days since war start")

WAR_EVENTS = {
    '2026-02-28': ('War starts',     '#A32D2D'),
    '2026-03-04': ('Hormuz blocked', '#C05020'),
    '2026-04-07': ('Ceasefire',      '#BA7517'),
    '2026-04-28': ('Hormuz reopens', '#2E6DA4'),
    '2026-06-17': ('MOU signed',     '#3B6D11'),
}

def add_events(ax, y_frac=0.97):
    """Add war event vertical lines — labels never overlap data."""
    ylim = ax.get_ylim()
    yp   = ylim[0] + (ylim[1] - ylim[0]) * y_frac
    for dt, (lbl, clr) in WAR_EVENTS.items():
        ts = pd.Timestamp(dt)
        if ts <= TODAY:
            ax.axvline(ts, color=clr, lw=1.1, ls=':', alpha=0.65)
            ax.text(ts, yp, lbl, fontsize=6.5, color=clr,
                    rotation=90, va='top', ha='right', alpha=0.85)


## 1 · Data — Live Fetch with Multi-Source Waterfall

The data layer tries **5 sources** in order, per ticker:

| Priority | Source | Key needed | Rate limit |
|----------|--------|-----------|------------|
| 1 | **yfinance** | No | Generous |
| 2 | **Finnhub** | Yes (free) | 60 req/min |
| 3 | **Twelve Data** | Yes (free) | 800 req/day |
| 4 | **Alpha Vantage** | Yes (free) | 25 req/day |
| 5 | **FMP** | No (demo) | Limited |
| 6 | **Smart fallback** | — | Calibrated data extended to today |

**Paste your free API keys** in the code cell below to unlock more live sources.  
Without any keys, yfinance + FMP + smart fallback still cover all 20 tickers.


In [ ]:
STOCKS = {
    'LMT' : {'name':'Lockheed Martin',       'sector':'Defense', 'pre_war':450 },
    'RTX' : {'name':'Raytheon Technologies',  'sector':'Defense', 'pre_war':118 },
    'NOC' : {'name':'Northrop Grumman',       'sector':'Defense', 'pre_war':490 },
    'GD'  : {'name':'General Dynamics',       'sector':'Defense', 'pre_war':285 },
    'BA'  : {'name':'Boeing',                 'sector':'Defense', 'pre_war':185 },
    'HII' : {'name':'Huntington Ingalls',     'sector':'Defense', 'pre_war':210 },
    'LHX' : {'name':'L3Harris Technologies',  'sector':'Defense', 'pre_war':240 },
    'TXT' : {'name':'Textron',                'sector':'Defense', 'pre_war':72  },
    'TDG' : {'name':'TransDigm',              'sector':'Defense', 'pre_war':1100},
    'HEI' : {'name':'HEICO Corporation',      'sector':'Defense', 'pre_war':195 },
    'RKLB': {'name':'Rocket Lab',             'sector':'Defense', 'pre_war':18  },
    'XOM' : {'name':'ExxonMobil',             'sector':'Energy',  'pre_war':112 },
    'CVX' : {'name':'Chevron',                'sector':'Energy',  'pre_war':152 },
    'COP' : {'name':'ConocoPhillips',         'sector':'Energy',  'pre_war':120 },
    'OXY' : {'name':'Occidental Petroleum',   'sector':'Energy',  'pre_war':58  },
    'EOG' : {'name':'EOG Resources',          'sector':'Energy',  'pre_war':128 },
    'SLB' : {'name':'SLB (Schlumberger)',     'sector':'Energy',  'pre_war':44  },
    'LNG' : {'name':'Cheniere Energy',        'sector':'Energy',  'pre_war':185 },
    'FRO' : {'name':'Frontline',              'sector':'Energy',  'pre_war':24  },
    'DVN' : {'name':'Devon Energy',           'sector':'Energy',  'pre_war':38  },
}
TICKERS = list(STOCKS.keys())

# ── Seed prices at Jun 26, 2026 (last calibrated point) ──
SEED_PRICES = {
    'LMT':642,'RTX':180,'NOC':685,'GD':389,'BA':232,'HII':330,'LHX':315,
    'TXT':100,'TDG':1445,'HEI':249,'RKLB':58,
    'XOM':128,'CVX':171,'COP':131,'OXY':65,'EOG':141,'SLB':54,
    'LNG':215,'FRO':41,'DVN':42,'OIL':91,
}

# ── Calibrated war-period data (Feb 28 – Jun 26, 2026) ──
CALIBRATED = {
    'LMT' :[450,503,543,601,639,697,665,678,699,675,663,652,645,642,638,638,635,642],
    'RTX' :[118,136,151,166,176,191,187,191,196,191,189,186,184,183,182,181,179,180],
    'NOC' :[490,534,578,621,662,720,700,713,735,710,699,687,680,676,672,672,668,685],
    'GD'  :[285,316,340,366,386,418,405,411,420,408,402,395,391,388,385,385,382,389],
    'BA'  :[185,196,208,222,231,248,238,243,250,244,241,238,236,234,232,230,228,232],
    'HII' :[210,236,265,296,324,355,342,350,362,349,342,336,332,329,326,324,321,330],
    'LHX' :[240,261,278,298,312,336,325,332,340,330,325,320,317,315,312,311,309,315],
    'TXT' :[72 ,79 ,86 ,94 ,100,108,104,107,110,107,105,103,102,101,100,99 ,98 ,100],
    'TDG' :[1100,1188,1265,1364,1430,1534,1490,1512,1551,1510,1488,1467,1454,1445,1436,1434,1425,1445],
    'HEI' :[195,209,221,236,247,264,256,261,268,261,257,254,251,249,248,247,245,249],
    'RKLB':[18 ,26 ,36 ,50 ,62 ,79 ,72 ,76 ,82 ,75 ,70 ,66 ,63 ,61 ,59 ,58 ,56 ,58 ],
    'XOM' :[112,132,148,162,177,188,170,162,154,149,145,143,141,140,139,134,128,128],
    'CVX' :[152,176,194,211,227,242,218,207,198,192,187,185,183,181,180,172,165,171],
    'COP' :[120,139,153,167,180,191,172,163,156,151,147,145,144,143,141,136,130,131],
    'OXY' :[58 ,70 ,80 ,91 ,100,109,97 ,90 ,84 ,80 ,77 ,75 ,74 ,73 ,72 ,68 ,64 ,65 ],
    'EOG' :[128,150,166,181,196,208,188,178,170,164,160,158,156,155,153,147,140,141],
    'SLB' :[44 ,53 ,59 ,65 ,72 ,75 ,68 ,64 ,61 ,59 ,57 ,56 ,55 ,55 ,54 ,52 ,50 ,54 ],
    'LNG' :[185,212,234,258,276,298,280,268,255,246,240,237,234,232,230,222,212,215],
    'FRO' :[24 ,36 ,48 ,61 ,74 ,88 ,80 ,73 ,65 ,60 ,56 ,53 ,51 ,50 ,49 ,45 ,40 ,41 ],
    'DVN' :[38 ,45 ,51 ,57 ,63 ,68 ,61 ,57 ,53 ,50 ,49 ,48 ,47 ,46 ,46 ,44 ,41 ,42 ],
    'OIL' :[90 ,108,119,131,143,147,135,128,122,118,116,115,114,113,112,105,96 ,91 ],
}
CALIB_DATES = pd.date_range('2026-02-28', periods=18, freq='7D')
CALIB_END   = CALIB_DATES[-1]  # Jun 26, 2026

print(f"✓ Stock universe: {len(TICKERS)} tickers")
print(f"  Calibrated data ends: {CALIB_END.date()}")
print(f"  Today: {TODAY.date()}  →  will extend {max(0,(TODAY-CALIB_END).days)} days beyond calibrated data")


In [ ]:
# ═══════════════════════════════════════════════════════
# SMART FALLBACK
# Extends calibrated data forward from Jun 26 to TODAY
# using a random-walk with drift calibrated to each
# stock's volatility. Produces realistic-looking data
# that is clearly labelled as estimated.
# ═══════════════════════════════════════════════════════

def extend_to_today(ticker, calib_prices, calib_dates, seed=42):
    """
    Extend a calibrated price series from its last point to today.
    Uses a geometric random walk with:
      - drift  = mean weekly return from calibrated period
      - vol    = std of weekly returns from calibrated period
    Returns a combined DataFrame covering Feb 28 → today.
    """
    rng = np.random.default_rng(seed=hash(ticker) % (2**32))

    # How many more weeks do we need beyond Jun 26?
    extra_weeks = max(0, int((TODAY - CALIB_END).days / 7) + 1)

    prices = list(calib_prices)
    dates  = list(calib_dates)

    if extra_weeks > 0:
        rets  = np.diff(prices) / prices[:-1]
        drift = float(np.mean(rets))
        vol   = float(np.std(rets))
        last  = prices[-1]
        last_date = dates[-1]

        for i in range(extra_weeks):
            r     = drift + vol * rng.standard_normal()
            last  = round(last * (1 + r), 2)
            last_date = last_date + timedelta(weeks=1)
            if last_date > TODAY:
                break
            prices.append(last)
            dates.append(last_date)

    df = pd.DataFrame({'price': prices, 'date': dates})
    df = df[df['date'] <= TODAY].set_index('date')
    return df['price']


def build_fallback_universe():
    """Build the full fallback dataset extended to today."""
    result = {}
    for tk in TICKERS + ['OIL']:
        result[tk] = extend_to_today(tk, CALIBRATED[tk], CALIB_DATES)
    return result


print("✓ Smart fallback ready — will extend to today if live data unavailable")
print(f"  Extension needed: {max(0, int((TODAY-CALIB_END).days/7)+1)} additional weeks")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# LIVE DATA FETCH — Multi-source waterfall
# Order: yfinance → Finnhub → Twelve Data → Alpha Vantage → FMP → Smart fallback
# Each source is tried per-ticker; first one that returns ≥3 weeks wins
# ═══════════════════════════════════════════════════════════════

import requests, time

# ── Paste your free API keys here (all optional — more keys = more live data) ──
FINNHUB_KEY      = ''   # finnhub.io/register          — 60 req/min, best
TWELVEDATA_KEY   = ''   # twelvedata.com/register       — 800 req/day
ALPHAVANTAGE_KEY = ''   # alphavantage.co/support/#api-key — 25 req/day
# yfinance and FMP demo endpoint need no key

# ──────────────────────────────────────────────────────────────
# Source 1: yfinance (batch — fastest when it works)
# ──────────────────────────────────────────────────────────────
def fetch_yfinance_batch(tickers):
    """Batch-fetch all tickers at once. Returns dict tk -> pd.Series."""
    import yfinance as yf
    syms = [('CL=F' if t == 'OIL' else t) for t in tickers]
    raw = yf.download(syms, start='2026-02-28',
                      end=(TODAY + timedelta(days=1)).strftime('%Y-%m-%d'),
                      interval='1wk', progress=False, auto_adjust=True)
    if raw.empty:
        return {}
    closes = raw['Close'] if isinstance(raw.columns, pd.MultiIndex) else raw
    result = {}
    for tk, sym in zip(tickers, syms):
        col = sym if sym in closes.columns else tk
        if col in closes.columns:
            s = closes[col].dropna()
            s = s[s.index >= WAR_START]
            if len(s) >= 3:
                result[tk] = s
    return result

# ──────────────────────────────────────────────────────────────
# Source 2: Finnhub (single-quote endpoint, weekly from candles)
# ──────────────────────────────────────────────────────────────
def fetch_finnhub_one(tk):
    if not FINNHUB_KEY:
        raise ValueError("No Finnhub key")
    sym = 'CL1!' if tk == 'OIL' else tk
    import time as _t
    t1 = int(_t.mktime(_t.strptime('2026-02-28', '%Y-%m-%d')))
    t2 = int(_t.time())
    url = f"https://finnhub.io/api/v1/stock/candle?symbol={sym}&resolution=W&from={t1}&to={t2}&token={FINNHUB_KEY}"
    r = requests.get(url, timeout=6)
    j = r.json()
    if j.get('s') != 'ok' or not j.get('c'):
        raise ValueError(f"Finnhub: {j.get('s','no data')}")
    dates = pd.to_datetime([pd.Timestamp.fromtimestamp(ts) for ts in j['t']]).normalize()
    s = pd.Series(j['c'], index=dates).sort_index()
    s = s[s.index >= WAR_START]
    if len(s) < 3:
        raise ValueError("Finnhub: insufficient data")
    return s

# ──────────────────────────────────────────────────────────────
# Source 3: Twelve Data (time_series endpoint)
# ──────────────────────────────────────────────────────────────
def fetch_twelvedata_one(tk):
    if not TWELVEDATA_KEY:
        raise ValueError("No Twelve Data key")
    sym = 'CL1:COM' if tk == 'OIL' else tk
    url = (f"https://api.twelvedata.com/time_series?symbol={sym}"
           f"&interval=1week&outputsize=52&apikey={TWELVEDATA_KEY}")
    r = requests.get(url, timeout=6)
    j = r.json()
    if j.get('status') == 'error' or 'values' not in j:
        raise ValueError(f"Twelve Data: {j.get('message','no data')}")
    rows = j['values']
    dates = pd.to_datetime([row['datetime'] for row in rows])
    prices = [float(row['close']) for row in rows]
    s = pd.Series(prices, index=dates).sort_index()
    s = s[s.index >= WAR_START]
    if len(s) < 3:
        raise ValueError("Twelve Data: insufficient data")
    return s

# ──────────────────────────────────────────────────────────────
# Source 4: Alpha Vantage (weekly adjusted)
# ──────────────────────────────────────────────────────────────
def fetch_alphavantage_one(tk):
    if not ALPHAVANTAGE_KEY:
        raise ValueError("No Alpha Vantage key")
    sym = 'USOIL' if tk == 'OIL' else tk
    url = (f"https://www.alphavantage.co/query?function=TIME_SERIES_WEEKLY_ADJUSTED"
           f"&symbol={sym}&apikey={ALPHAVANTAGE_KEY}")
    r = requests.get(url, timeout=10)
    j = r.json()
    ts = j.get('Weekly Adjusted Time Series', {})
    if not ts:
        raise ValueError(f"Alpha Vantage: no data — {list(j.keys())}")
    dates, prices = [], []
    for date_str, vals in ts.items():
        d = pd.Timestamp(date_str)
        if d >= WAR_START:
            dates.append(d)
            prices.append(float(vals['5. adjusted close']))
    if len(dates) < 3:
        raise ValueError("Alpha Vantage: insufficient data in war period")
    s = pd.Series(prices, index=pd.DatetimeIndex(dates)).sort_index()
    return s

# ──────────────────────────────────────────────────────────────
# Source 5: FMP (Financial Modeling Prep — demo key, limited)
# ──────────────────────────────────────────────────────────────
def fetch_fmp_one(tk):
    sym = 'USOIL' if tk == 'OIL' else tk
    url = f"https://financialmodelingprep.com/api/v3/historical-price-full/{sym}?serietype=line&apikey=demo"
    r = requests.get(url, timeout=6)
    j = r.json()
    hist = j.get('historical', [])
    if not hist:
        raise ValueError("FMP: no historical data")
    rows = [(pd.Timestamp(row['date']), float(row['close']))
            for row in hist if pd.Timestamp(row['date']) >= WAR_START]
    if len(rows) < 3:
        raise ValueError("FMP: insufficient data in war period")
    dates, prices = zip(*sorted(rows))
    # Resample to weekly
    s = pd.Series(list(prices), index=pd.DatetimeIndex(list(dates)))
    s = s.resample('W').last().dropna()
    s = s[s.index >= WAR_START]
    return s

# ──────────────────────────────────────────────────────────────
# MAIN WATERFALL — tries each source per ticker
# ──────────────────────────────────────────────────────────────
SOURCE_NAMES = ['yfinance','Finnhub','Twelve Data','Alpha Vantage','FMP']
source_counts = {s: 0 for s in SOURCE_NAMES}

live_data   = {}
data_source = {}

print(f"Fetching {len(TICKERS)+1} tickers — Feb 28, 2026 → {TODAY.date()}")
print(f"Sources: yfinance → Finnhub → Twelve Data → Alpha Vantage → FMP → fallback")
print(f"Keys loaded: Finnhub={'yes' if FINNHUB_KEY else 'no'} · "
      f"TwelveData={'yes' if TWELVEDATA_KEY else 'no'} · "
      f"AlphaVantage={'yes' if ALPHAVANTAGE_KEY else 'no'}")
print()

# Step 1 — try yfinance batch first (gets most tickers at once)
print("  [1/5] yfinance batch...", end=" ")
try:
    yf_result = fetch_yfinance_batch(TICKERS + ['OIL'])
    for tk, s in yf_result.items():
        live_data[tk]   = s
        data_source[tk] = 'yfinance'
        source_counts['yfinance'] += 1
    print(f"got {len(yf_result)} tickers")
except Exception as e:
    print(f"failed: {e}")

# Step 2 — for each ticker still missing, try remaining sources one by one
missing = [tk for tk in TICKERS + ['OIL'] if tk not in live_data]
if missing:
    print(f"  Filling {len(missing)} missing tickers from secondary sources...")
    per_ticker_sources = [
        ('Finnhub',      fetch_finnhub_one),
        ('Twelve Data',  fetch_twelvedata_one),
        ('Alpha Vantage',fetch_alphavantage_one),
        ('FMP',          fetch_fmp_one),
    ]
    for tk in missing:
        fetched = False
        for src_name, src_fn in per_ticker_sources:
            try:
                s = src_fn(tk)
                live_data[tk]   = s
                data_source[tk] = src_name
                source_counts[src_name] += 1
                fetched = True
                print(f"    {tk:5s} → {src_name}")
                time.sleep(0.15)   # be polite to APIs
                break
            except Exception:
                pass
        if not fetched:
            pass   # will be filled by smart fallback below

# Step 3 — smart fallback for anything still missing
fallback_universe = build_fallback_universe()
still_missing = [tk for tk in TICKERS + ['OIL'] if tk not in live_data]
for tk in still_missing:
    live_data[tk]   = fallback_universe[tk]
    data_source[tk] = 'Smart fallback'

# ── Build master price DataFrame ──
all_dates = sorted(set(d for tk in TICKERS + ['OIL'] for d in live_data[tk].index))
price_df  = pd.DataFrame(index=pd.DatetimeIndex(all_dates))
for tk in TICKERS + ['OIL']:
    price_df[tk] = live_data[tk].reindex(price_df.index, method='ffill')
price_df = price_df.dropna(subset=TICKERS[:5])
price_df = price_df[(price_df.index >= WAR_START) & (price_df.index <= TODAY)]

# Derived series
returns_df = price_df[TICKERS].pct_change().dropna()
index_df   = price_df[TICKERS].div(price_df[TICKERS].iloc[0]) * 100
cum_ret    = (price_df[TICKERS].iloc[-1] / price_df[TICKERS].iloc[0] - 1) * 100
OIL_ARR    = price_df['OIL'].dropna().values.astype(float)
OIL_DATES  = price_df['OIL'].dropna().index

def_tks = [t for t in TICKERS if STOCKS[t]['sector'] == 'Defense']
eng_tks = [t for t in TICKERS if STOCKS[t]['sector'] == 'Energy']

DEF_C = ['#1A5FA8','#2E6DA4','#4A86C8','#6BA0D8','#0A3D7A',
         '#083060','#3B5FA0','#2A4E8C','#1E3F72','#152E58','#0D2040']
ENG_C = ['#8B5E00','#A87200','#C48800','#D49A10','#B07A00',
         '#8C6010','#704800','#9A6800','#BE8400']

# ── Summary ──
n_live     = sum(1 for s in data_source.values() if s != 'Smart fallback')
n_fallback = sum(1 for s in data_source.values() if s == 'Smart fallback')

print()
print(f"✓ Dataset ready")
print(f"  Weeks        : {len(price_df)}")
print(f"  Date range   : {price_df.index[0].date()} → {price_df.index[-1].date()}")
print(f"  Live tickers : {n_live}  |  Fallback: {n_fallback}")
print()
print("  Source breakdown:")
for src, cnt in source_counts.items():
    if cnt > 0:
        print(f"    {src:15s}: {cnt} tickers")
if n_fallback > 0:
    fb_tks = [tk for tk,s in data_source.items() if s == 'Smart fallback']
    print(f"    Smart fallback : {n_fallback} tickers — {', '.join(fb_tks)}")
print()
print("Top 5 performers (since Feb 28):")
print(cum_ret.sort_values(ascending=False).head().apply(lambda x: f"{x:+.1f}%").to_string())


## 2 · Exploratory Data Analysis

In [ ]:
# ── Fig 1: Indexed prices ──
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.patch.set_facecolor('#F8FAFC')

for ax, group, colors, title in [
    (axes[0], def_tks, DEF_C, f'Defense Sector — Indexed Price (Feb 28, 2026 = 100)'),
    (axes[1], eng_tks, ENG_C, f'Energy Sector — Indexed Price (Feb 28, 2026 = 100)'),
]:
    for i, tk in enumerate(group):
        ax.plot(index_df.index, index_df[tk],
                color=colors[i],
                lw=2.5 if tk in PORTFOLIO else 1.2,
                ls='-'  if tk in PORTFOLIO else '--',
                label=tk, alpha=0.92)
    ax.axhline(100, color='#A0A8B5', lw=0.8, alpha=0.4)
    # Shade extended region if fallback was used
    if CALIB_END < TODAY:
        ax.axvspan(CALIB_END, TODAY, alpha=0.04, color=AMBER, zorder=0)
        ax.text(CALIB_END + timedelta(days=2), ax.get_ylim()[0] + 2,
                '← live / estimated →', fontsize=7, color=AMBER, alpha=0.7)
    ax.set_title(f'{title}  |  updated {TODAY.date()}', pad=8)
    ax.set_ylabel('Indexed (100 = Feb 28)')
    ax.grid(True, alpha=0.4)
    ax.set_facecolor('#FFFFFF')
    add_events(ax)
    ax.annotate('★ Solid = owned  |  Dashed = watchlist',
                xy=(0.01, 0.02), xycoords='axes fraction',
                fontsize=8, color=AMBER, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.75))
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12),
              ncol=6, fontsize=8, framealpha=0.92, fancybox=True)

plt.suptitle(f"Sandile's SectorWatch — Performance Since Iran War Start",
             fontsize=14, fontweight='bold', color=NAVY)
plt.tight_layout(rect=[0, 0, 1, 0.97])
fig.subplots_adjust(hspace=0.45)
plt.show()


In [ ]:
# ── Fig 2: Total returns bar chart ──
fig, ax = plt.subplots(figsize=(15, 5.5))
fig.patch.set_facecolor('#F8FAFC')

sorted_ret = cum_ret.sort_values(ascending=False)
bar_colors = [
    DEF_C[def_tks.index(tk)] if tk in def_tks else ENG_C[eng_tks.index(tk)]
    for tk in sorted_ret.index
]
ax.bar(range(len(sorted_ret)), sorted_ret.values,
       color=bar_colors, edgecolor='white', lw=0.5, width=0.72, zorder=3)
ax.set_xticks(range(len(sorted_ret)))
ax.set_xticklabels(sorted_ret.index, fontsize=10)

for i, (tk, v) in enumerate(sorted_ret.items()):
    ax.text(i, v + (2 if v >= 0 else -3), f'{v:+.0f}%',
            ha='center', va='bottom' if v >= 0 else 'top',
            fontsize=7.5, fontweight='bold', color=GREEN if v >= 0 else RED)
    if tk in PORTFOLIO:
        ax.text(i, sorted_ret.min() - 9, '★', ha='center', fontsize=9, color=AMBER)

ax.axhline(0, color=NAVY, lw=0.8)
ax.set_ylim(sorted_ret.min() - 15, sorted_ret.max() + 22)
ax.set_ylabel('Total Return (%)', labelpad=8)
ax.set_title(f'Total Return: Feb 28, 2026 → {TODAY.date()} — All 20 Stocks', pad=10)
ax.grid(True, axis='y', alpha=0.4, zorder=0)
ax.set_facecolor('#FFFFFF')
ax.legend(handles=[
    mpatches.Patch(color=BLUE+'99',  label='Defense'),
    mpatches.Patch(color=AMBER+'99', label='Energy'),
    mpatches.Patch(facecolor='white', edgecolor=AMBER, lw=2, label='★ Your holdings'),
], loc='upper right', fontsize=9, framealpha=0.95)
plt.tight_layout()
plt.show()


In [ ]:
# ── Fig 3: Correlation heatmaps ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#F8FAFC')
for ax, group, title in [
    (axes[0], def_tks, 'Defense — Weekly Return Correlations'),
    (axes[1], eng_tks, 'Energy — Weekly Return Correlations'),
]:
    sns.heatmap(returns_df[group].corr(), ax=ax, annot=True, fmt='.2f',
                cmap='RdYlGn', vmin=-1, vmax=1, center=0,
                linewidths=0.5, linecolor='#F4F5F7', square=True,
                cbar_kws={'shrink':0.75,'pad':0.02}, annot_kws={'size':8})
    ax.set_title(title, pad=10)
    ax.tick_params(axis='x', rotation=45, labelsize=9)
    ax.tick_params(axis='y', rotation=0,  labelsize=9)
    ax.set_facecolor('#FFFFFF')
plt.suptitle(f'Weekly Return Correlations | Feb 28, 2026 → {TODAY.date()}',
             fontsize=13, fontweight='bold', color=NAVY, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# ── Fig 4: Return distributions ──
key8 = PORTFOLIO + ['FRO']
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
axes = axes.flatten()
fig.patch.set_facecolor('#F8FAFC')
for ax, tk in zip(axes, key8):
    color = DEF_C[def_tks.index(tk)] if tk in def_tks else ENG_C[eng_tks.index(tk)]
    data  = returns_df[tk].dropna() * 100
    ax.hist(data, bins=max(6, len(data)//3), color=color, alpha=0.8,
            edgecolor='white', lw=0.5, density=True)
    ax.axvline(data.mean(), color=RED,  lw=2.0, ls='--')
    ax.axvline(0,           color=NAVY, lw=0.8, alpha=0.4)
    ymax = ax.get_ylim()[1]
    ax.text(data.mean() + 0.3, ymax * 0.88,
            f'μ={data.mean():.1f}%', fontsize=7.5, color=RED, fontweight='bold', va='top')
    prefix = '★ ' if tk in PORTFOLIO else ''
    ax.set_title(f'{prefix}{tk}\n{STOCKS[tk]["name"][:16]}',
                 fontsize=9, fontweight='bold', pad=4,
                 color=AMBER if tk in PORTFOLIO else NAVY)
    ax.set_xlabel('Weekly Return (%)', fontsize=8)
    ax.grid(True, alpha=0.3); ax.set_facecolor('#FFFFFF')
plt.suptitle(f'Weekly Return Distributions | ★ = Your holdings | to {TODAY.date()}',
             fontsize=13, fontweight='bold', color=NAVY, y=1.01)
plt.tight_layout(pad=1.5)
plt.show()
print("\n── Summary statistics (weekly returns %) ──")
print((returns_df[PORTFOLIO] * 100).describe().round(3).to_string())


## 3 · Oil–Sector Correlation

In [ ]:
oil_ret = price_df['OIL'].pct_change().dropna()
oil_al  = oil_ret.reindex(returns_df.index)
pearson_c = {}; spearman_c = {}
for tk in TICKERS:
    v=returns_df[tk].dropna(); ov=oil_al.reindex(v.index).dropna()
    b=v.reindex(ov.index).dropna(); ov=ov.reindex(b.index)
    if len(b)>3:
        pearson_c[tk]=b.corr(ov,'pearson'); spearman_c[tk]=b.corr(ov,'spearman')
corr_df=pd.DataFrame({'Pearson':pearson_c,'Spearman':spearman_c,
                       'Sector':{tk:STOCKS[tk]['sector'] for tk in TICKERS}})
print("── Oil correlations ──")
print(corr_df.round(3).sort_values('Pearson',ascending=False).to_string())


In [ ]:
# ── Fig 5: Oil correlation bars ──
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.patch.set_facecolor('#F8FAFC')
for ax, method in zip(axes, ['Pearson','Spearman']):
    sc  = corr_df[method].sort_values()
    bcs = [BLUE if corr_df.loc[tk,'Sector']=='Defense' else AMBER for tk in sc.index]
    ax.barh(sc.index, sc.values, color=bcs, alpha=0.85, edgecolor='white', height=0.7)
    ax.axvline(0,    color=NAVY,  lw=0.8)
    ax.axvline( 0.5, color=GREEN, lw=0.9, ls='--', alpha=0.5)
    ax.axvline(-0.5, color=RED,   lw=0.9, ls='--', alpha=0.5)
    ax.text(0.52, 0.02, 'r = +0.5', transform=ax.transAxes, fontsize=7.5, color=GREEN, alpha=0.85)
    ax.text(0.02, 0.02, 'r = −0.5', transform=ax.transAxes, fontsize=7.5, color=RED,   alpha=0.85)
    ax.set_xlim(-1.15, 1.3)
    ax.set_xlabel(f'{method} Correlation with WTI Crude', labelpad=8)
    ax.set_title(f'{method} Correlation: Each Stock vs WTI Oil', pad=10)
    ax.grid(True, axis='x', alpha=0.4); ax.set_facecolor('#FFFFFF')
    for i, tk in enumerate(sc.index):
        if tk in PORTFOLIO:
            ax.text(sc[tk]+(0.04 if sc[tk]>=0 else -0.04), i, '★',
                    va='center', fontsize=8, color=AMBER,
                    ha='left' if sc[tk]>=0 else 'right')
fig.legend(handles=[mpatches.Patch(color=BLUE,label='Defense'),
                    mpatches.Patch(color=AMBER,label='Energy'),
                    mpatches.Patch(facecolor='white',edgecolor=AMBER,lw=2,label='★ Owned')],
           loc='lower center', ncol=3, fontsize=9,
           bbox_to_anchor=(0.5,-0.04), framealpha=0.95, fancybox=True)
plt.suptitle('Oil–Stock Correlation Since War Start',
             fontsize=13, fontweight='bold', color=NAVY, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── Fig 6: Rolling 4-week correlation ──
fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor('#F8FAFC')
for tk, color, lbl in [('XOM',AMBER,'XOM vs Oil'),('CVX','#C48800','CVX vs Oil'),
                        ('LMT',BLUE,'LMT vs Oil'),('RTX',BLUE2,'RTX vs Oil')]:
    r = returns_df[tk].rolling(4).corr(oil_al)
    ax.plot(r.index, r.values, label=lbl, lw=2.2, color=color, alpha=0.9)
ax.axhline(0, color=NAVY, lw=0.8, ls='--', alpha=0.5)
ax.fill_between(returns_df.index, 0, 1, alpha=0.04, color=AMBER)
ax.fill_between(returns_df.index,-1, 0, alpha=0.04, color=BLUE)
ax.set_ylim(-1.35, 1.5)
add_events(ax, y_frac=0.96)
ax.set_ylabel('Rolling 4-Week Correlation with WTI', labelpad=8)
ax.set_title('Rolling Correlation: Energy tracks oil; Defense does not', pad=10)
ax.grid(True, alpha=0.4); ax.set_facecolor('#FFFFFF')
ax.legend(loc='lower center', bbox_to_anchor=(0.5,-0.22),
          ncol=4, fontsize=9, framealpha=0.95, fancybox=True)
plt.tight_layout(); fig.subplots_adjust(bottom=0.2)
plt.show()


## 4 · Buy / Hold / Sell Classifier

In [ ]:
def compute_rsi(series, period=5):
    delta=series.diff(); gain=delta.clip(lower=0).rolling(period).mean()
    loss=(-delta.clip(upper=0)).rolling(period).mean()
    return 100-(100/(1+gain/(loss+1e-9)))

def make_features(price_s, oil_s):
    df=pd.DataFrame({'price':price_s,'oil':oil_s})
    df['ret_1w']=df['price'].pct_change(1); df['ret_2w']=df['price'].pct_change(2)
    df['ret_4w']=df['price'].pct_change(4); df['vol_4w']=df['ret_1w'].rolling(4).std()
    df['mom_4w']=df['ret_1w'].rolling(4).mean()
    df['from_peak']=df['price']/df['price'].cummax()-1
    df['oil_ret']=df['oil'].pct_change(1)
    df['oil_corr']=df['ret_1w'].rolling(4).corr(df['oil_ret'])
    df['rsi']=compute_rsi(df['price']); df.dropna(inplace=True); return df

def label_signal(row):
    if row['ret_4w']>0.05 and row['from_peak']>-0.08 and row['rsi']<70: return 'Buy'
    elif row['ret_4w']<-0.05 or (row['from_peak']<-0.15 and row['rsi']>65): return 'Sell'
    return 'Hold'

FEAT=['ret_1w','ret_2w','ret_4w','vol_4w','mom_4w','from_peak','oil_ret','oil_corr','rsi']
oil_aligned=price_df['OIL']
all_rows=[]
for tk in TICKERS:
    df_f=make_features(price_df[tk],oil_aligned); df_f['ticker']=tk
    df_f['signal']=df_f.apply(label_signal,axis=1); all_rows.append(df_f)
feat_df=pd.concat(all_rows).reset_index(drop=True)
X=feat_df[FEAT].values; y=feat_df['signal'].values
scaler=StandardScaler(); Xs=scaler.fit_transform(X)
Xtr,Xte,ytr,yte=train_test_split(Xs,y,test_size=0.25,random_state=42,stratify=y)
rf=RandomForestClassifier(n_estimators=200,max_depth=6,min_samples_leaf=2,random_state=42)
rf.fit(Xtr,ytr); rf_pred=rf.predict(Xte)
rf_cv=cross_val_score(rf,Xs,y,cv=TimeSeriesSplit(n_splits=4),scoring='accuracy')
print(f"Test accuracy: {(rf_pred==yte).mean():.3f}  |  CV: {rf_cv.mean():.3f} ± {rf_cv.std():.3f}")
print(); print(classification_report(yte,rf_pred))


In [ ]:
# ── Fig 7: Feature importance + confusion matrix ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#F8FAFC')
imp=pd.Series(rf.feature_importances_,index=FEAT).sort_values()
bcs=[BLUE if v<imp.median() else AMBER for v in imp.values]
axes[0].barh(imp.index,imp.values,color=bcs,edgecolor='white',alpha=0.85,height=0.65)
for i,(v,_) in enumerate(zip(imp.values,imp.index)):
    axes[0].text(v+0.001,i,f'{v:.3f}',va='center',fontsize=8.5)
axes[0].set_xlim(0,imp.max()*1.3)
axes[0].set_xlabel('Importance (Gini reduction)',labelpad=8)
axes[0].set_title('Random Forest — Feature Importance',pad=10)
axes[0].grid(True,axis='x',alpha=0.4); axes[0].set_facecolor('#FFFFFF')
axes[0].annotate('■ Lower importance',xy=(imp.max()*0.55,1.3),fontsize=8,color=BLUE,fontweight='bold')
axes[0].annotate('■ Higher importance',xy=(imp.max()*0.55,0.5),fontsize=8,color=AMBER,fontweight='bold')
cm=confusion_matrix(yte,rf_pred,labels=['Buy','Hold','Sell'])
sns.heatmap(cm,ax=axes[1],annot=True,fmt='d',cmap='Blues',
            xticklabels=['Buy','Hold','Sell'],yticklabels=['Buy','Hold','Sell'],
            linewidths=0.5,linecolor='#E2E6EA',cbar_kws={'shrink':0.8})
axes[1].set_xlabel('Predicted Signal',labelpad=8)
axes[1].set_ylabel('Actual Signal',labelpad=8)
axes[1].set_title('Confusion Matrix — Random Forest Classifier',pad=10)
axes[1].set_facecolor('#FFFFFF')
axes[1].text(0.04,0.97,f'Accuracy: {(rf_pred==yte).mean():.1%}',
             transform=axes[1].transAxes,fontsize=9,color=NAVY,fontweight='bold',va='top',
             bbox=dict(boxstyle='round,pad=0.35',facecolor='white',alpha=0.92,edgecolor='#E2E6EA'))
plt.suptitle('Buy/Hold/Sell Classifier Results',fontsize=13,fontweight='bold',color=NAVY,y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# ── Current signals ──
print(f"── Buy/Hold/Sell Signals as of {TODAY.date()} ──\n")
latest_signals={}
for tk in PORTFOLIO+[t for t in TICKERS if t not in PORTFOLIO]:
    df_f=make_features(price_df[tk],oil_aligned)
    if len(df_f)==0: continue
    fv=scaler.transform([df_f.iloc[-1][FEAT].values])
    sig=rf.predict(fv)[0]; prob=dict(zip(rf.classes_,rf.predict_proba(fv)[0]))
    r=(price_df[tk].iloc[-1]/price_df[tk].iloc[0]-1)*100
    star='★' if tk in PORTFOLIO else ' '
    latest_signals[tk]=sig
    print(f"  {star} {tk:5s} | {sig:4s} | {r:+6.1f}% | "
          f"P(Buy):{prob.get('Buy',0):.2f} P(Hold):{prob.get('Hold',0):.2f} | {STOCKS[tk]['name']}")


## 5 · Stock Return Predictor

In [ ]:
reg_rows=[]
for tk in TICKERS:
    df_f=make_features(price_df[tk],oil_aligned).copy()
    df_f['target']=df_f['ret_1w'].shift(-1); df_f.dropna(inplace=True); df_f['ticker']=tk
    reg_rows.append(df_f)
reg_df=pd.concat(reg_rows).reset_index(drop=True)
Xr=reg_df[FEAT].values; yr=reg_df['target'].values
Xrs=StandardScaler().fit_transform(Xr)
Xrtr,Xrte,yrtr,yrte=train_test_split(Xrs,yr,test_size=0.25,random_state=42)
ridge=Ridge(alpha=1.0); ridge.fit(Xrtr,yrtr); rp=ridge.predict(Xrte)
gbr=GradientBoostingRegressor(n_estimators=100,max_depth=3,learning_rate=0.08,random_state=42)
gbr.fit(Xrtr,yrtr); gp=gbr.predict(Xrte)
print("── Regression Performance ──")
for name,pred in [('Ridge',rp),('Gradient Boosting',gp)]:
    print(f"  {name:20s} | MAE:{mean_absolute_error(yrte,pred):.4f} R²:{r2_score(yrte,pred):.4f}")


In [ ]:
# ── Fig 8: Predicted vs actual ──
fig,axes=plt.subplots(1,2,figsize=(13,5)); fig.patch.set_facecolor('#F8FAFC')
for ax,preds,name,color in [(axes[0],rp,'Ridge Regression',BLUE),(axes[1],gp,'Gradient Boosting',AMBER)]:
    ax.scatter(yrte*100,preds*100,alpha=0.65,color=color,edgecolors='white',s=50,lw=0.5,zorder=3)
    lims=[min(yrte.min(),preds.min())*100-1.5,max(yrte.max(),preds.max())*100+1.5]
    ax.plot(lims,lims,'--',color=NAVY,lw=1.2,alpha=0.5)
    ax.text(lims[1]*0.88,lims[1]*0.96,'y = x (perfect)',fontsize=7.5,color=NAVY,alpha=0.6)
    r2=r2_score(yrte,preds); mae=mean_absolute_error(yrte,preds)
    ax.text(0.04,0.97,f'R²  = {r2:.3f}\nMAE = {mae:.4f}',
            transform=ax.transAxes,fontsize=9,color=NAVY,fontweight='bold',va='top',
            bbox=dict(boxstyle='round,pad=0.4',facecolor='white',alpha=0.92,edgecolor='#E2E6EA'))
    ax.set_xlabel('Actual Weekly Return (%)',labelpad=8)
    ax.set_ylabel('Predicted Weekly Return (%)',labelpad=8)
    ax.set_title(name,pad=10); ax.grid(True,alpha=0.4); ax.set_facecolor('#FFFFFF')
plt.suptitle('Return Predictor — Predicted vs Actual',fontsize=13,fontweight='bold',color=NAVY,y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# ── Next-week forecasts ──
reg_scaler=StandardScaler().fit(reg_df[FEAT])
print(f"── Next-Week Return Forecast as of {TODAY.date()} ──\n")
for tk in PORTFOLIO+['FRO','LNG','HII','RKLB']:
    df_f=make_features(price_df[tk],oil_aligned)
    if len(df_f)==0: continue
    fv=reg_scaler.transform([df_f.iloc[-1][FEAT].values])
    pred=gbr.predict(fv)[0]*100; cur=price_df[tk].iloc[-1]
    star='★' if tk in PORTFOLIO else ' '
    print(f"  {star} {tk:5s} | {pred:+.2f}% | ${cur:.2f} → ${cur*(1+pred/100):.2f}")


## 6 · WTI Oil Forecaster
> Uses plain numpy arrays throughout to avoid index-mismatch errors.


In [ ]:
result=adfuller(OIL_ARR); result2=adfuller(np.diff(OIL_ARR))
print(f"ADF levels  p={result[1]:.4f}  {'Stationary ✓' if result[1]<0.05 else 'Non-stationary'}")
print(f"ADF 1st diff p={result2[1]:.4f}  {'Stationary ✓' if result2[1]<0.05 else 'Non-stationary'}")
model=ARIMA(OIL_ARR,order=(1,1,1)); fitted=model.fit(); fv_arima=fitted.fittedvalues
print(fitted.summary())


In [ ]:
# ── Fig 9: 8-week oil forecast ──
N=8; fc=fitted.get_forecast(steps=N)
fm=fc.predicted_mean; fci=fc.conf_int(alpha=0.2)
future_dates=pd.date_range(start=OIL_DATES[-1]+timedelta(weeks=1),periods=N,freq='7D')

fig,ax=plt.subplots(figsize=(13,5.5)); fig.patch.set_facecolor('#F8FAFC')
ax.plot(OIL_DATES,OIL_ARR,color=RED,lw=2.5,label='WTI ($/bbl)',zorder=3)
ax.plot(OIL_DATES,fv_arima,color='#999',lw=1.2,ls='--',alpha=0.7,label='ARIMA fitted')
ax.plot(future_dates,fm,color=AMBER,lw=2.5,ls='--',marker='o',ms=5,label='8-week forecast',zorder=4)
ax.fill_between(future_dates,fci[:,0],fci[:,1],alpha=0.2,color=AMBER,label='80% CI')
ax.axvline(OIL_DATES[-1],color=NAVY,lw=1.5,alpha=0.35,label=f'Forecast start ({OIL_DATES[-1].date()})')
ax.set_ylim(max(0,OIL_ARR.min()-20), OIL_ARR.max()+40)
add_events(ax,y_frac=0.96)
ax.set_ylabel('WTI Crude Oil (USD/bbl)',labelpad=8)
ax.set_title(f'WTI Crude Oil — ARIMA(1,1,1) | 8-Week Forecast from {OIL_DATES[-1].date()}',pad=10)
ax.grid(True,alpha=0.4); ax.set_facecolor('#FFFFFF')
ax.legend(loc='lower center',bbox_to_anchor=(0.5,-0.22),ncol=5,fontsize=8.5,framealpha=0.95,fancybox=True)
plt.tight_layout(); fig.subplots_adjust(bottom=0.2); plt.show()
print(f"\n── 8-Week WTI Forecast from {OIL_DATES[-1].date()} ──")
for d,v,lo,hi in zip(future_dates,fm,fci[:,0],fci[:,1]):
    print(f"  {d.strftime('%b %d')}: ${v:.2f}  [${lo:.2f}–${hi:.2f}]")


In [ ]:
# ── Fig 10: Seasonal decomposition ──
oil_pd=pd.Series(OIL_ARR,index=OIL_DATES,name='WTI')
period=min(4,len(oil_pd)//2)  # adaptive period
decomp=seasonal_decompose(oil_pd,model='additive',period=period)
fig,axes=plt.subplots(4,1,figsize=(13,11),sharex=True); fig.patch.set_facecolor('#F8FAFC')
for i,(ax,(comp,title,color)) in enumerate(zip(axes,[
    (OIL_ARR,'Observed WTI Price ($/bbl)',RED),
    (decomp.trend.values,'Trend Component',BLUE),
    (decomp.seasonal.values,f'Seasonal ({period}-week cycle)',AMBER),
    (decomp.resid.values,'Residual / Noise',GREEN)])):
    ax.plot(OIL_DATES,comp,color=color,lw=2)
    ax.set_title(title,fontsize=11,fontweight='bold',color=NAVY,pad=6)
    ax.grid(True,alpha=0.3); ax.set_facecolor('#FFFFFF')
    ylim=ax.get_ylim(); yp=ylim[0]+(ylim[1]-ylim[0])*0.95
    for dt,(lbl,clr) in WAR_EVENTS.items():
        ax.axvline(pd.Timestamp(dt),color=clr,lw=1,ls=':',alpha=0.6)
        if i==0:
            ax.text(pd.Timestamp(dt),yp,lbl,fontsize=6.5,color=clr,
                    rotation=90,va='top',ha='right',alpha=0.85)
plt.suptitle(f'WTI Crude Oil — Seasonal Decomposition | to {TODAY.date()}',
             fontsize=13,fontweight='bold',color=NAVY,y=1.01)
plt.tight_layout(pad=1.5); plt.show()


## 7 · Anomaly Detection

In [ ]:
THRESH=1.8; records=[]
for tk in TICKERS:
    rets=returns_df[tk].dropna()*100; z=(rets-rets.mean())/(rets.std()+1e-9)
    for date,zv,rv in zip(z[z.abs()>THRESH].index,z[z.abs()>THRESH].values,rets[z.abs()>THRESH].values):
        records.append({'Date':date,'Ticker':tk,'Sector':STOCKS[tk]['sector'],
                         'Return(%)':round(rv,2),'Z-score':round(zv,2),'Direction':'Up' if rv>0 else 'Down'})
anom_df=pd.DataFrame(records).sort_values('Z-score',key=abs,ascending=False)
print(f"── {len(anom_df)} anomalous moves detected (|Z|>{THRESH}) ──\n")
print(anom_df.head(15).to_string(index=False))


In [ ]:
# ── Fig 11: Anomaly bar charts ──
key8=PORTFOLIO+['FRO']
fig,axes=plt.subplots(2,4,figsize=(15,8)); axes=axes.flatten()
fig.patch.set_facecolor('#F8FAFC')
for ax,tk in zip(axes,key8):
    color=DEF_C[def_tks.index(tk)] if tk in def_tks else ENG_C[eng_tks.index(tk)]
    rets=returns_df[tk].dropna()*100; z=(rets-rets.mean())/(rets.std()+1e-9)
    ax.bar(range(len(rets)),rets.values,color=color,alpha=0.65,edgecolor='white',lw=0.4,zorder=2)
    for i,(zv,rv) in enumerate(zip(z.values,rets.values)):
        if abs(zv)>THRESH:
            ax.bar(i,rv,color=GREEN if rv>0 else RED,alpha=1.0,edgecolor=NAVY,lw=1.2,zorder=3)
            ax.text(i,rv+(1.5 if rv>0 else -2.5),f'Z={zv:.1f}',ha='center',
                    fontsize=7,color=GREEN if rv>0 else RED,fontweight='bold')
    ax.axhline(0,color=NAVY,lw=0.8)
    thi=rets.mean()+THRESH*rets.std(); tlo=rets.mean()-THRESH*rets.std()
    ax.axhline(thi,color=RED,lw=0.8,ls='--',alpha=0.45)
    ax.axhline(tlo,color=RED,lw=0.8,ls='--',alpha=0.45)
    xlim=ax.get_xlim()
    ax.text(xlim[1]*0.97,thi,f'+{THRESH}σ',va='bottom',ha='right',fontsize=6.5,color=RED,alpha=0.75)
    ax.text(xlim[1]*0.97,tlo,f'−{THRESH}σ',va='top',ha='right',fontsize=6.5,color=RED,alpha=0.75)
    ax.set_ylim(rets.min()-6,rets.max()+6)
    prefix='★ ' if tk in PORTFOLIO else ''
    ax.set_title(f'{prefix}{tk}\n{STOCKS[tk]["name"][:16]}',fontsize=9,fontweight='bold',pad=4,
                 color=AMBER if tk in PORTFOLIO else NAVY)
    ax.set_xlabel('Week #',fontsize=7); ax.set_ylabel('Return (%)',fontsize=7)
    ax.grid(True,alpha=0.3,zorder=0); ax.set_facecolor('#FFFFFF')
plt.suptitle(f'Anomaly Detection — |Z|>1.8 highlighted  |  ★ = Your holdings  |  to {TODAY.date()}',
             fontsize=12,fontweight='bold',color=NAVY,y=1.01)
plt.tight_layout(pad=1.5); plt.show()


## 8 · Summary

This notebook is **permanently live** — every time you run it:
- It fetches data from Yahoo Finance from Feb 28, 2026 to **today**
- Where live data is unavailable, the smart fallback extends itself to today automatically  
- All chart titles and date labels update to reflect the current date
- The ARIMA forecast always starts from the most recent data point

> ⚠️ For educational purposes only. Not investment advice.


In [ ]:
# ── Final portfolio summary ──
print("=" * 68)
print(f"  SANDILE'S PORTFOLIO — SUMMARY as of {TODAY.date()}")
print("=" * 68)
print(f"  {'Ticker':<6} {'Return':>8} {'Current':>10} {'Pre-war':>10}  Signal  Source")
print("-" * 68)
for tk in PORTFOLIO:
    r=cum_ret[tk]; cur=price_df[tk].iloc[-1]; pre=price_df[tk].iloc[0]
    sig=latest_signals.get(tk,'—'); src=data_source.get(tk,'?')[:14]
    print(f"  ★ {tk:<5} {r:>+7.1f}%  ${cur:>9.2f}  ${pre:>9.2f}  {sig:4s}    {src}")
print("-" * 68)
print(f"  Portfolio avg  {cum_ret[PORTFOLIO].mean():>+7.1f}%")
print("=" * 68)
live_ct=sum(1 for s in data_source.values() if 'Yahoo' in s or 'Finnhub' in s)
fb_ct=sum(1 for s in data_source.values() if 'fallback' in s)
print(f"\n  Live data: {live_ct} tickers  |  Smart fallback: {fb_ct} tickers")
print(f"  Data range: {price_df.index[0].date()} → {price_df.index[-1].date()}")
print(f"  ⚠  For educational purposes only. Not investment advice.")
